# Silver Layer

### Creating Silver Tables for the first time

In [2]:
# source_schema = "Bronze"
# target_schema = "Silver"

# tables = spark.catalog.listTables(source_schema)

# for table in tables:
#     table_name = table.name

#     df = spark.read.table(f"{source_schema}.{table_name}")

#     # Create empty DataFrame with the same Schema
#     empty_df = df.limit(0)

#     (
#         empty_df.write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable(f"{target_schema}.{table_name}")
#     )

# print("Empty Silver tables created successfully.")

StatementMeta(, 89593433-bc2f-4023-836c-ef9337a18301, 4, Finished, Available, Finished, False)

Empty Silver tables created successfully.


### Incremental Data Extraction Function

In [1]:
import pyspark.sql.functions as f

def get_incremental_data(table_name):
    """
    Reads Bronze and Silver tables, retrieves the latest load timestamp
    from the Silver table, and returns only the new records from Bronze.
    """

    # Read Bronze table
    bronze_df = spark.read.table(f"Bronze.{table_name}")

    # Read Silver table
    silver_df = spark.read.table(f"Silver.{table_name}")

    # Get the latest load timestamp from Silver
    last_load = (
        silver_df
        .agg(f.max("load_datetime").alias("last_load"))
        .collect()[0]["last_load"]
    )

    # First load -> return all Bronze records
    if last_load is None:
        return bronze_df

    # Incremental load -> return only new records
    return bronze_df.filter(
        f.col("load_datetime") > last_load
    )

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 3, Finished, Available, Finished, False)

### Validation Function

In [2]:
def validate_dataframe(df, table_name, primary_key):
    """
    Validate a cleaned DataFrame before merging into the Silver layer.

    Parameters
    ----------
    df : DataFrame
        Cleaned Spark DataFrame.
    table_name : str
        Table name used for reporting.
    primary_key : str
        Primary key column.
    """

    print("=" * 60)
    print(f"{table_name} Validation Report")
    print("=" * 60)

    # Row Count
    row_count = df.count()
    print(f"Total Rows: {row_count}")

    # Duplicate Primary Keys
    duplicate_count = (
        df.groupBy(primary_key)
          .count()
          .filter(f.col("count") > 1)
          .count()
    )

    print(f"Duplicate {primary_key}: {duplicate_count}")

    # Null Values
    print("\nNull Values Per Column:")

    display(
        df.select(
            [
                f.count(
                    f.when(f.col(col).isNull(), col)
                ).alias(col)
                for col in df.columns
            ]
        )
    )

    # Sample Data
    print("\nSample Data:")

    display(df.limit(5))

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 4, Finished, Available, Finished, False)

### Merge Data into Silver Function

In [14]:
from delta.tables import DeltaTable

def merge_to_silver(table_name, clean_df, primary_key):
    """
    Merge cleaned data into the Silver table.

    Parameters
    ----------
    table_name : str
        Silver table name.
    clean_df : DataFrame
        Cleaned dataframe to merge.
    primary_key : str
        Primary key column used for matching.
    """

    silver_table = DeltaTable.forName(
        spark,
        f"Silver.{table_name}"
    )

    merge_condition = (
        f"tgt.{primary_key} = src.{primary_key}"
    )

    update_columns = {
        col: f"src.{col}"
        for col in clean_df.columns
    }

    (
        silver_table.alias("tgt")
        .merge(
            clean_df.alias("src"),
            merge_condition
        )
        .whenMatchedUpdate(set=update_columns)
        .whenNotMatchedInsert(values=update_columns)
        .execute()
    )
    print(f"Incremental {table_name} To Silver Done")

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 16, Finished, Available, Finished, False)

### Appointment

In [17]:
# Get Appointments Incremental Data
appointments_df = get_incremental_data("appointments")
display(appointments_df.limit(5))

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 59f03b2b-03cc-459b-9c30-e9fabede7e00)

In [18]:
# Clean Appointment Incremental
from pyspark.sql.functions import initcap

clean_appointments = (
    appointments_df
    .withColumn("appointment_id", f.trim(f.col("appointment_id")))
    .withColumn("patient_id", f.trim(f.col("patient_id")))
    .withColumn("doctor_id", f.trim(f.col("doctor_id")))
    .withColumn("clinic_id", f.trim(f.col("clinic_id")))

    .withColumn("status", f.upper(f.trim(f.col("status"))))

    .withColumn(
        "reason_for_visit",
        initcap(f.trim(f.col("reason_for_visit")))
    )

    .withColumn(
        "appointment_date",
        f.to_date(f.col("appointment_date"), "MM/dd/yyyy")
    )

    .withColumn(
        "load_datetime",
        f.to_timestamp(f.col("load_datetime"))
    )
    .withColumn(
        "source_file",
        f.regexp_extract(
            "source_file",
            r"([^/]+\.csv)",
            1
        )
    )
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 20, Finished, Available, Finished, False)

In [19]:
# Validate Appointments Incremental Data
validate_dataframe(clean_appointments,"appointments","appointment_id")

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 21, Finished, Available, Finished, False)

appointments Validation Report
Total Rows: 100000
Duplicate appointment_id: 0

Null Values Per Column:


SynapseWidget(Synapse.DataFrame, f396bddf-df8c-4e54-8b1b-b7677fe2a27a)


Sample Data:


SynapseWidget(Synapse.DataFrame, 99c7f12a-aa1f-4024-af36-9109342d2d18)

In [20]:
# Merge Appointment Data Into Silver Table
merge_to_silver("appointments",clean_appointments,"appointment_id")

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 22, Finished, Available, Finished, False)

Incremental appointments To Silver Done


### Patient

In [21]:
# Get Patients Incremental Data
patients_df = get_incremental_data("patients")
display(patients_df.limit(5))

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0e0436a0-9126-4966-88b3-96fbb978b91d)

In [22]:
# Clean Patients Incremental
cleaned_patients = (
    patients_df
    .withColumn("patient_id", f.trim("patient_id"))
    .withColumn("full_name", f.initcap(f.trim("full_name")))
    .withColumn("gender", f.initcap(f.trim("gender")))
    .withColumn("date_of_birth", f.to_date("date_of_birth", "MM/dd/yyyy"))
    .withColumn("phone", f.trim("phone"))
    .withColumn("email", f.lower(f.trim("email")))
    .withColumn("address", f.trim("address"))
    .withColumn("registration_date", f.to_date("registration_date", "MM/dd/yyyy"))
    .withColumn("blood_group", f.upper(f.trim("blood_group")))
    .withColumn("emergency_contact", f.trim("emergency_contact"))
    .withColumn("load_datetime", f.to_timestamp("load_datetime"))
    .withColumn(
        "source_file",
        f.regexp_extract(
            "source_file",
            r"([^/]+\.csv)",
            1
        )
    )
    .dropna(subset=["patient_id"])
    .dropDuplicates(["patient_id"])
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 24, Finished, Available, Finished, False)

In [23]:
# Validate Patients Incremental Data
validate_dataframe(cleaned_patients, "Patients", "patient_id")

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 25, Finished, Available, Finished, False)

Patients Validation Report
Total Rows: 20000
Duplicate patient_id: 0

Null Values Per Column:


SynapseWidget(Synapse.DataFrame, 593d48d2-7004-4ca1-8a94-557abd37789b)


Sample Data:


SynapseWidget(Synapse.DataFrame, aeba521f-e73c-484d-95ac-362eb3e4bcf3)

In [24]:
# Insert Patients Data Into Silver Table
merge_to_silver("patients",cleaned_patients,"patient_id")

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 26, Finished, Available, Finished, False)

Incremental patients To Silver Done


### Clinics

In [25]:
# Get Clinics Incremental Data
clinics_df = get_incremental_data("clinics")
display(clinics_df.limit(5))

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 84a43ae7-9bc3-46f2-93ec-ea227491d62a)

In [26]:
# Clean Clinics Incremental Data
cleaned_clinics = (
    clinics_df
    .withColumn("clinic_id", f.trim("clinic_id"))
    .withColumn("clinic_name", f.initcap(f.trim("clinic_name")))
    .withColumn("address", f.trim("address"))
    .withColumn("city", f.initcap(f.trim("city")))
    .withColumn("state", f.initcap(f.trim("state")))
    .withColumn("phone", f.trim("phone"))
    .withColumn("opening_hours", f.trim("opening_hours"))
    .withColumn("clinic_type", f.initcap(f.trim("clinic_type")))
    .withColumn("load_datetime", f.to_timestamp("load_datetime"))
    .withColumn(
        "source_file",
        f.regexp_extract(
            "source_file",
            r"([^/]+\.csv)",
            1
        )
    )
    .dropna(subset=["clinic_id"])
    .dropDuplicates(["clinic_id"])
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 28, Finished, Available, Finished, False)

In [27]:
# Validate Clinics Incremental Data
validate_dataframe(
    cleaned_clinics,
    "Clinics",
    "clinic_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 29, Finished, Available, Finished, False)

Clinics Validation Report
Total Rows: 50
Duplicate clinic_id: 0

Null Values Per Column:


SynapseWidget(Synapse.DataFrame, a52f7295-ce27-4e27-86d7-2eeb1d567aa8)


Sample Data:


SynapseWidget(Synapse.DataFrame, 60a30dac-5ad5-4a09-b146-530af063beec)

In [28]:
# Merge Clinics Data Into Silver Table
merge_to_silver(
    table_name="clinics",
    clean_df=cleaned_clinics,
    primary_key="clinic_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 30, Finished, Available, Finished, False)

Incremental clinics To Silver Done


### Doctors

In [29]:
# Get Doctors Incremental Data
doctors_df = get_incremental_data("doctors")
display(doctors_df.limit(5))

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4bad1143-c973-4b53-a2cc-3ee67da85367)

In [30]:
# Clean Doctors Incremental Data
clean_doctors_df = (
    doctors_df
    .withColumn("doctor_id", f.trim("doctor_id"))
    .withColumn("full_name", f.initcap(f.trim("full_name")))
    .withColumn("specialty", f.initcap(f.trim("specialty")))
    .withColumn("phone", f.trim("phone"))
    .withColumn("email", f.lower(f.trim("email")))
    .withColumn("clinic_id", f.trim("clinic_id"))
    .withColumn("load_datetime", f.to_timestamp("load_datetime"))
    .withColumn(
        "source_file",
        f.regexp_extract(
            "source_file",
            r"([^/]+\.csv)",
            1
        )
    )
    .dropna(subset=["doctor_id"])
    .dropDuplicates(["doctor_id"])
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 32, Finished, Available, Finished, False)

In [31]:
# Validate Doctors Incremental Data
validate_dataframe(
    clean_doctors_df,
    "Doctors",
    "doctor_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 33, Finished, Available, Finished, False)

Doctors Validation Report
Total Rows: 500
Duplicate doctor_id: 0

Null Values Per Column:


SynapseWidget(Synapse.DataFrame, 90dfb9c3-f8ba-42a5-8ca0-515eb79efdfb)


Sample Data:


SynapseWidget(Synapse.DataFrame, 7e1cbb0f-9fa1-4baa-8329-5106bed7aead)

In [32]:
# Merge Doctors Data Into Silver Table
merge_to_silver(
    table_name="doctors",
    clean_df=clean_doctors_df,
    primary_key="doctor_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 34, Finished, Available, Finished, False)

Incremental doctors To Silver Done


### Insurance

In [33]:
# Get Insurance Incremental Data
insurance_df = get_incremental_data("insurance")
display(insurance_df.limit(5))

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc8a71b1-be3b-4d55-a451-083e0cb355db)

In [34]:
# Clean Insurance Incremental Data
clean_insurance_df = (
    insurance_df
    .withColumn("patient_id", f.trim("patient_id"))
    .withColumn("insurance_status", f.initcap(f.trim("insurance_status")))
    .withColumn("insurance_provider", f.initcap(f.trim("insurance_provider")))
    .withColumn("coverage_level", f.initcap(f.trim("coverage_level")))
    .withColumn("load_datetime", f.to_timestamp("load_datetime"))
    .withColumn(
        "source_file",
        f.regexp_extract(
            "source_file",
            r"([^/]+\.csv)",
            1
        )
    )
    .dropna(subset=["patient_id"])
    .dropDuplicates(["patient_id"])
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 36, Finished, Available, Finished, False)

In [35]:
# Validate Insurance Incremental Data
validate_dataframe(
    clean_insurance_df,
    "Insurance",
    "patient_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 37, Finished, Available, Finished, False)

Insurance Validation Report
Total Rows: 20000
Duplicate patient_id: 0

Null Values Per Column:


SynapseWidget(Synapse.DataFrame, d5c8de9d-5936-4842-b1f9-92598d23975c)


Sample Data:


SynapseWidget(Synapse.DataFrame, b1c06639-e88f-4e5e-8d53-bc4d94cbc8ea)

In [36]:
# Merge Insurance Data Into Silver Table
merge_to_silver(
    table_name="insurance",
    clean_df=clean_insurance_df,
    primary_key="patient_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 38, Finished, Available, Finished, False)

Incremental insurance To Silver Done


### Payments

In [37]:
# Get Payments Incremental Data
payments_df = get_incremental_data("payments")
display(payments_df.limit(5))

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 69a09bdb-2c5b-46c5-8d55-0e34fb5026b6)

In [38]:
# Clean Payments Incremental Data
from pyspark.sql.types import DecimalType

clean_payments_df = (
    payments_df
    .withColumn("payment_id", f.trim("payment_id"))
    .withColumn("appointment_id", f.trim("appointment_id"))
    .withColumn("payment_date", f.to_date("payment_date", "MM/dd/yyyy"))
    .withColumn("payment_amount", f.col("payment_amount").cast(DecimalType(10, 2)))
    .withColumn("payment_method", f.initcap(f.trim("payment_method")))
    .withColumn("payment_status", f.initcap(f.trim("payment_status")))
    .withColumn("load_datetime", f.to_timestamp("load_datetime"))
    .withColumn(
        "source_file",
        f.regexp_extract(
            "source_file",
            r"([^/]+\.csv)",
            1
        )
    )
    .dropna(subset=["payment_id"])
    .dropDuplicates(["payment_id"])
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 40, Finished, Available, Finished, False)

In [39]:
# Validate Payments Incremental Data
validate_dataframe(
    clean_payments_df,
    "Payments",
    "payment_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 41, Finished, Available, Finished, False)

Payments Validation Report
Total Rows: 100000
Duplicate payment_id: 0

Null Values Per Column:


SynapseWidget(Synapse.DataFrame, 072cad8e-921e-481f-ae94-8d6d835cb02c)


Sample Data:


SynapseWidget(Synapse.DataFrame, 77942ab8-6a6a-4e3a-ab4a-8ed2456b597e)

In [40]:
# Merge Payments Data Into Silver Table
merge_to_silver(
    table_name="payments",
    clean_df=clean_payments_df,
    primary_key="payment_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 42, Finished, Available, Finished, False)

Incremental payments To Silver Done


### Prescriptions

In [41]:
# Get Prescriptions Incremental Data
prescriptions_df = get_incremental_data("prescriptions")
display(prescriptions_df.limit(5))

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa7a94b0-2d8d-48be-a2d0-506e941f9357)

In [42]:
# Clean Prescriptions Incremental Data
clean_prescriptions_df = (
    prescriptions_df
    .withColumn("prescription_id", f.trim("prescription_id"))
    .withColumn("appointment_id", f.trim("appointment_id"))
    .withColumn("medicine_name", f.initcap(f.trim("medicine_name")))
    .withColumn("instructions", f.trim("instructions"))
    .withColumn("load_datetime", f.to_timestamp("load_datetime"))
    .withColumn(
        "source_file",
        f.regexp_extract(
            "source_file",
            r"([^/]+\.csv)",
            1
        )
    )
    .dropna(subset=["prescription_id"])
    .dropDuplicates(["prescription_id"])
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 44, Finished, Available, Finished, False)

In [43]:
# Validate Prescriptions Incremental Data
validate_dataframe(
    clean_prescriptions_df,
    "Prescriptions",
    "prescription_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 45, Finished, Available, Finished, False)

Prescriptions Validation Report
Total Rows: 100000
Duplicate prescription_id: 0

Null Values Per Column:


SynapseWidget(Synapse.DataFrame, 42be29be-37f5-4a54-9bb7-ffc2695f87dd)


Sample Data:


SynapseWidget(Synapse.DataFrame, 17b8a8d8-f7ec-4e8d-bbe6-bfd911291012)

In [44]:
# Merge Prescriptions Data Into Silver Table
merge_to_silver(
    table_name="prescriptions",
    clean_df=clean_prescriptions_df,
    primary_key="prescription_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 46, Finished, Available, Finished, False)

Incremental prescriptions To Silver Done


### Treatments

In [45]:
# Get Treatments Incremental Data
treatments_df = get_incremental_data("treatments")
display(treatments_df.limit(5))

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eee9c811-9a18-40de-ab85-6bd37aeaf81d)

In [46]:
# Clean Treatments Incremental Data
from pyspark.sql.types import DecimalType

clean_treatments_df = (
    treatments_df
    .withColumn("treatment_id", f.trim("treatment_id"))
    .withColumn("appointment_id", f.trim("appointment_id"))
    .withColumn("treatment_type", f.initcap(f.trim("treatment_type")))
    .withColumn("treatment_cost", f.col("treatment_cost").cast(DecimalType(10, 2)))
    .withColumn("treatment_status", f.initcap(f.trim("treatment_status")))
    .withColumn("load_datetime", f.to_timestamp("load_datetime"))
    .withColumn(
        "source_file",
        f.regexp_extract(
            "source_file",
            r"([^/]+\.csv)",
            1
        )
    )
    .dropna(subset=["treatment_id"])
    .dropDuplicates(["treatment_id"])
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 48, Finished, Available, Finished, False)

In [47]:
# Validate Treatments Incremental Data
validate_dataframe(
    clean_treatments_df,
    "Treatments",
    "treatment_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 49, Finished, Available, Finished, False)

Treatments Validation Report
Total Rows: 100000
Duplicate treatment_id: 0

Null Values Per Column:


SynapseWidget(Synapse.DataFrame, 50e0fd6f-579e-410c-9adf-6dd0d567eaa4)


Sample Data:


SynapseWidget(Synapse.DataFrame, 1e930343-1b2b-48d5-bafd-007414c040f4)

In [48]:
# Merge Treatments Data Into Silver Table
merge_to_silver(
    table_name="treatments",
    clean_df=clean_treatments_df,
    primary_key="treatment_id"
)

StatementMeta(, 0a6b29bc-a4d6-44d3-984b-9fcf55fae054, 50, Finished, Available, Finished, False)

Incremental treatments To Silver Done
